In [1]:
!git clone https://acharyayush@github.com/yogeshY0/Signature-Verification.git main

Cloning into 'main'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 57 (delta 24), reused 49 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 176.78 KiB | 11.79 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [2]:
cd main

/kaggle/working/main


In [3]:
!pip install -q timm opencv-python-headless
!ls /kaggle/input/datasets/acharyayush/signature-dataset-cedar-bhsig260
!python -m unittest discover -s tests -t .

BHSig260  cedar  manifest.csv
..............
----------------------------------------------------------------------
Ran 14 tests in 2.926s

OK


In [4]:
DATA="/kaggle/input/datasets/acharyayush/signature-dataset-cedar-bhsig260"
!python -m training.train_siamese --manifest $DATA/manifest.csv --already-preprocessed \
  --output-dir /kaggle/working/ckpt_smoke --device cuda --workers 2 \
  --phase1-epochs 1 --phase2-epochs 1

Using device: cuda

model.safetensors: downloading bytes: ████████▍             | 43.6MB, 1.79MB/s  
model.safetensors: downloading bytes: ██████████████▊       | 77.0MB, 4.28MB/s  
model.safetensors: reconstructing file:  28%|█▋    | 32.3MB /  114MB, 2.70MB/s  
model.safetensors: downloading bytes: ████████████████████  |  104MB, 9.64MB/s  
model.safetensors: downloading bytes: ██████████████████████|  109MB, 10.1MB/s  
model.safetensors: reconstructing file: 100%|██████|  114MB /  114MB, 10.8MB/s  
{"phase": "warmup", "epoch": 1, "train": {"loss": 0.277648936761053, "mean_positive": 0.7527743797553214, "mean_negative": 0.7751254564837405, "active_fraction": 1.0}, "validation": {"loss": 0.25252269208431244, "mean_positive": 0.41971778124570847, "mean_negative": 0.4772430434823036, "active_fraction": 0.9062500149011612}}
{"phase": "fine_tune", "epoch": 1, "train": {"loss": 0.3483836368510598, "mean_positive": 0.6310390265364396, "mean_negative": 0.582655414154655, "active_fraction": 1.

In [5]:
# Baseline: current balanced batch (12 signers, 4 genuine, 2 forged per signer).
!python -m training.train_siamese --manifest $DATA/manifest.csv --already-preprocessed \
  --output-dir /kaggle/working/ckpt_f2 --device cuda --workers 2 \
  --phase1-epochs 5 --phase2-epochs 20 \
  --signers-per-batch 12 --genuine-per-signer 4 --forgeries-per-signer 2


Using device: cuda
{"phase": "warmup", "epoch": 1, "train": {"loss": 0.277648936761053, "mean_positive": 0.7527743797553214, "mean_negative": 0.7751254564837405, "active_fraction": 1.0}, "validation": {"loss": 0.25252269208431244, "mean_positive": 0.41971778124570847, "mean_negative": 0.4772430434823036, "active_fraction": 0.9062500149011612}}
{"phase": "warmup", "epoch": 2, "train": {"loss": 0.26479321404507283, "mean_positive": 0.7329652748609844, "mean_negative": 0.768192215969688, "active_fraction": 0.9978070196352506}, "validation": {"loss": 0.2247776910662651, "mean_positive": 0.4544002562761307, "mean_negative": 0.5504661053419113, "active_fraction": 0.8125000149011612}}
{"phase": "warmup", "epoch": 3, "train": {"loss": 0.24644159251137784, "mean_positive": 0.764146553842645, "mean_negative": 0.8179260680550023, "active_fraction": 0.9923245655862909}, "validation": {"loss": 0.23561490699648857, "mean_positive": 0.48920053243637085, "mean_negative": 0.5810324251651764, "active_fr

In [ ]:
# Comparison: same training settings, but 4 forged samples per signer.
# Compare ckpt_f4 against ckpt_f2 with the evaluation cell below.
!python -m training.train_siamese --manifest $DATA/manifest.csv --already-preprocessed \
  --output-dir /kaggle/working/ckpt_f4 --device cuda --workers 2 \
  --phase1-epochs 5 --phase2-epochs 20 \
  --signers-per-batch 12 --genuine-per-signer 4 --forgeries-per-signer 4


In [6]:
for CKPT in ckpt_f2 ckpt_f4; do
  python -m training.evaluate --checkpoint /kaggle/working/$CKPT/best_model.pt \
    --manifest $DATA/manifest.csv --already-preprocessed --bootstrap 1000 \
    > /kaggle/working/${CKPT}_evaluation.json
done
cat /kaggle/working/ckpt_f2_evaluation.json
cat /kaggle/working/ckpt_f4_evaluation.json


{
  "threshold_from_validation": 0.8235858082771301,
  "validation_eer_skilled": 0.12233713677764568,
  "test": {
    "far_skilled_at_threshold": 0.07864357864357864,
    "far_random_at_threshold": 0.009611795446061963,
    "frr_at_threshold": 0.12171052631578948,
    "eer_skilled": 0.09876490468595732,
    "eer_random": 0.03179824561403509,
    "genuine_queries": 912,
    "skilled_forgeries": 1386,
    "random_pairs": 42864
  }
}
